In [1]:
import pyrootutils
root = pyrootutils.setup_root(
    search_from='/root/zhaokj/CVLface/cvlface/research/recognition/code/work_0925/ijbc.ipynb',
    indicator=["__root__.txt"],
    pythonpath=True,
    dotenv=True,
)
import os, sys
sys.path.append(os.path.join(root))
import numpy as np
np.bool = np.bool_  # fix bug for mxnet 1.9.1
np.object = np.object_

import pandas as pd
from models import get_model
from aligners import get_aligner
from evaluations import get_evaluator_by_name
from lightning.fabric.loggers import CSVLogger
from pipelines import pipeline_from_name
from lightning.pytorch.loggers import WandbLogger
from general_utils.config_utils import load_config
from evaluations import summary
from lightning.fabric import Fabric
from functools import partial
from fabric.fabric import setup_dataloader_from_dataset
from tqdm import tqdm
import lovely_tensors as lt
lt.monkey_patch()
import pickle

def get_runname_and_task(ckpt_dir):
    if 'pretrained_models' in ckpt_dir:
        runname = ckpt_dir.split('/')[-1]
        code_task = os.path.abspath('/root/zhaokj/CVLface/cvlface/research/recognition/code/work_0925/ijbc.ipynb').split('/')[-2]
        save_dir_task = 'pretrained_models'
    else:
        runname = ckpt_dir.split('/')[-3]
        code_task = os.path.abspath('/root/zhaokj/CVLface/cvlface/research/recognition/code/work_0925/ijbc.ipynb').split('/')[-2]
        save_dir_task = code_task
    return runname, save_dir_task, code_task


In [ ]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--num_gpu', type=int, default=1)
parser.add_argument('--precision', type=str, default='bf16-mixed')
parser.add_argument('--eval_config_name', type=str, default='ijbc')
parser.add_argument('--pipeline_name', type=str, default='default')
parser.add_argument('--ckpt_dir', type=str, default="/root/zhaokj/CVLface/cvlface/research/recognition/experiments/work_0925/ft_ir101_s3_full_09-25_0/checkpoints_every_epoch/epoch:29_step:293580")
# parser.add_argument('--save_path', type=str, default='')
# parser.add_argument('--image_dir', type=str, default='')

# parser.add_argument('--save_path', type=str, default='/root/zhaokj/work/fp_fn/0730_val')
parser.add_argument('--save_path', type=str, default='./output')
parser.add_argument('--image_dir', type=str, default='/data1/dataset/test')
args = parser.parse_args([])

In [ ]:
runname, save_dir_task, task = get_runname_and_task(args.ckpt_dir)
eval_config = load_config(f'evaluations/configs/{args.eval_config_name}.yaml')
output_dir = os.path.join(root, 'research/recognition/experiments', save_dir_task, 'eval_' + runname)
os.makedirs(output_dir, exist_ok=True)


In [ ]:
model_config = load_config(os.path.join(args.ckpt_dir, 'model.yaml'))
model = get_model(model_config, task)
model.load_state_dict_from_path(os.path.join(args.ckpt_dir, 'model.pt'))
train_transform = model.make_train_transform()
test_transform = model.make_test_transform()


In [ ]:
if os.path.exists(os.path.join(args.ckpt_dir, 'aligner.yaml')):
    aligner_config = load_config(os.path.join(args.ckpt_dir, 'aligner.yaml'))
    aligner_config.start_from = os.path.join(args.ckpt_dir, 'aligner.pt')
    aligner = get_aligner(aligner_config)
else:
    aligner_config = load_config(os.path.join(root, 'research/recognition/code/', task, f'aligners/configs/none.yaml'))
    aligner = get_aligner(aligner_config)
aligner

In [ ]:
if args.pipeline_name == 'default':
    full_config_path = os.path.join(args.ckpt_dir, 'config.yaml')
    assert os.path.isfile(full_config_path), f"config.yaml not found at {full_config_path}, try with pipeline name"
    pipeline_name = load_config(full_config_path).pipelines.eval_pipeline_name
else:
    pipeline_name = args.pipeline_name
pipeline_name

In [ ]:
fabric = Fabric(precision='bf16-mixed',
                    accelerator="auto",
                    devices=1,
                    )

In [ ]:
fabric.launch()

In [ ]:
fabric.setup_dataloader_from_dataset = partial(setup_dataloader_from_dataset, fabric=fabric, seed=2048)
model = fabric.setup(model)

In [ ]:
eval_pipeline = pipeline_from_name(pipeline_name, model, aligner)
eval_pipeline.integrity_check(dataset_color_space='RGB')
eval_pipeline

In [ ]:
info = eval_config['per_epoch_evaluations']['IJBC_gt_aligned']
info

In [ ]:
eval_data_path = os.path.join(eval_config.data_root, info.path)
eval_data_path

In [ ]:
eval_type = info.evaluation_type
eval_type

In [ ]:
from datasets import Dataset
import torch
from functools import partial

dataset = Dataset.load_from_disk(eval_data_path)

In [ ]:
def preprocess_transform(examples, image_transforms):
    images = [image.convert("RGB") for image in examples['image']]
    images = [image_transforms(image) for image in images]
    examples["pixel_values"] = images
    return examples
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()

    indexes = torch.tensor([example["index"] for example in examples], dtype=torch.int)

    return {
        "pixel_values": pixel_values,
        "index": indexes,
    }

In [ ]:
transform=eval_pipeline.make_test_transform()
preprocess = partial(preprocess_transform, image_transforms=transform)
dataset = dataset.with_transform(preprocess)

In [ ]:
dataloader = fabric.setup_dataloader_from_dataset(dataset,is_train=False,
                                                               batch_size=32,
                                                               num_workers=1,
                                                               collate_fn=collate_fn)

In [ ]:
meta = torch.load(os.path.join(eval_data_path, 'metadata.pt'), weights_only=False)

In [ ]:
meta

In [ ]:
for batch_idx, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    print(batch_idx)
    print(batch)
    break

In [ ]:
all_features = []
all_index = []
flip_images = False
for batch_idx, batch in tqdm(enumerate(dataloader), total=len(dataloader), desc='IJB Feature',
                                disable=fabric.local_rank != 0):

    images = batch['pixel_values']
    index = batch['index']

    if flip_images:
        images = torch.flip(images, dims=[3])
    features = eval_pipeline(images)
    all_features.append(features.cpu().detach())
    all_index.append(index.cpu().detach())

In [ ]:
torch.cuda.empty_cache()

In [ ]:
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/epoch48_features.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
embeddings = (data['collection']['features'] + data['collection']['features']).numpy()

In [2]:
import torch
meta = torch.load(os.path.join('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned', 'metadata.pt'), weights_only=False)

In [ ]:
infernece_configs = [{'use_norm_score': True, 'use_detector_score': True},
                         {'use_norm_score': True, 'use_detector_score': False},
                         {'use_norm_score': False, 'use_detector_score': True}, ]


In [ ]:
use_norm_score = False
use_detector_score = False
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))

In [3]:
len(meta['templates'])

469375

In [4]:
meta['templates'][:20], meta['medias'][:10]


(array([1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]),
 array([69544,  3720,     3,     1,  3726, 69607,  3727,  7907,  7915,
        33727]))

In [5]:
meta['p1'][:10],meta['p2'][:10]

(array([1, 1, 1, 1, 1, 1, 2, 2, 2, 2]),
 array([11065, 11066, 11067, 11068, 11069, 11070, 11071, 11072, 11073,
        11074]))

In [6]:
meta['label'][:10]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [7]:
# 有 469375 张图片, 每个图片对应一个tempalte,多个template可能对应同一人,
# 图片序号与template关系:
index_template_dict = {i: meta['templates'][i] for i in range(len(meta['templates']))}


In [ ]:
# 并查集获取所有template的唯一id，包括孤岛
from collections import defaultdict

# 收集所有唯一节点
all_nodes = set(meta['p1']) | set(meta['p2'])

class UnionFind:
    def __init__(self, nodes):
        self.parent = {x: x for x in nodes}
        self.rank = {x: 0 for x in nodes}
    
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1

# 初始化并查集
uf = UnionFind(all_nodes)

# 合并 label == 1 的边
for label, p1, p2 in zip(meta['label'], meta['p1'], meta['p2']):
    if label == 1:
        uf.union(p1, p2)

# 生成 group_map（包含孤岛）
group_map = {}
root_to_id = {}
for node in all_nodes:
    root = uf.find(node)
    if root not in root_to_id:
        root_to_id[root] = len(root_to_id)  # 自动递增 group_id
    group_map[node] = root_to_id[root]